# Two-Stage Text Models

This notebook keeps the project in notebook form while reusing the same baseline model code in `src/`.

- Stage 1 trains on all rows with `Dialogue -> Manipulative`
- Stage 2 trains on manipulative rows only with `Dialogue -> exact Technique string`

The helper scripts are the backend; this notebook is the analysis/report layer.


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebooks'
PROJECT_ROOT = NOTEBOOK_DIR.parent

sys.path.append(str(PROJECT_ROOT))

RESULTS_DIR = NOTEBOOK_DIR / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
FIGURES_DIR = NOTEBOOK_DIR / 'figures'

for path in [METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

import pandas as pd
from IPython.display import display

from src.load_data import DEFAULT_CONFIG, build_binary_dataframe, build_technique_dataframe, load_raw_dataframe
from src.majority_baseline import print_summary as print_majority_summary, run_majority_baseline, save_metrics as save_majority_metrics, save_predictions as save_majority_predictions
from src.naivebayes import print_summary as print_nb_summary, run_multinomial_nb, save_baseline_comparison_chart, save_classification_report, save_confusion_matrix_plot, save_metrics as save_nb_metrics, save_predictions as save_nb_predictions


## Load MentalManip Stage Data


In [2]:
raw_df = load_raw_dataframe(config=DEFAULT_CONFIG)
stage1_df = build_binary_dataframe(raw_df)
stage2_df = build_technique_dataframe(raw_df)

print(f'Stage 1 rows: {len(stage1_df)}')
print(f'Stage 2 rows before optional rare-label handling: {len(stage2_df)}')


Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Stage 1 rows: 4000
Stage 2 rows before optional rare-label handling: 2154


## Stage 1: Majority Baseline


In [3]:
stage1_majority_metrics, stage1_majority_predictions, stage1_majority_report = run_majority_baseline(stage1_df, stage_name='stage1_binary')
save_majority_metrics(stage1_majority_metrics, output_path=METRICS_DIR / 'stage1_binary_majority_baseline.json')
save_majority_predictions(stage1_majority_predictions, output_path=PREDICTIONS_DIR / 'stage1_binary_majority_baseline_predictions.csv')
stage1_majority_report.to_csv(METRICS_DIR / 'stage1_binary_majority_baseline_report.csv', index=False)
print_majority_summary(stage1_majority_metrics)


Majority Baseline Summary
Stage: stage1_binary
Total examples: 4000
Train size / test size: 3200 / 800
Number of classes: 2
Majority class: manipulative
Accuracy: 0.7050
Macro F1: 0.4135
Weighted F1: 0.5830
Precision: 0.7050
Recall: 1.0000
F1: 0.8270


In [4]:
display(stage1_majority_report.head(10))


,class,precision,recall,f1_score,support
0,manipulative,0.705,1.0,0.826979,564
1,non_manipulative,0.000,0.0,0.000000,236


## Stage 1: TF-IDF + Multinomial Naive Bayes


In [5]:
stage1_nb_metrics, stage1_nb_predictions, stage1_nb_report = run_multinomial_nb(stage1_df, stage_name='stage1_binary', debug=False)
save_nb_metrics(stage1_nb_metrics, output_path=METRICS_DIR / 'stage1_binary_text_only_multinomial_nb.json')
save_nb_predictions(stage1_nb_predictions, output_path=PREDICTIONS_DIR / 'stage1_binary_text_only_multinomial_nb_predictions.csv')
save_classification_report(stage1_nb_report, output_path=METRICS_DIR / 'stage1_binary_text_only_multinomial_nb_report.csv')
save_confusion_matrix_plot(stage1_nb_metrics['confusion_matrix'], output_path=FIGURES_DIR / 'stage1_binary_text_only_multinomial_nb_confusion_matrix.png', title='Stage 1 Binary Confusion Matrix')
save_baseline_comparison_chart(stage1_nb_metrics, majority_metrics_path=METRICS_DIR / 'stage1_binary_majority_baseline.json', output_path=FIGURES_DIR / 'stage1_binary_model_comparison.png')
print_nb_summary(stage1_nb_metrics)


TF-IDF + Multinomial Naive Bayes Summary
Stage: stage1_binary
Total examples: 4000
Train size / test size: 3200 / 800
Accuracy: 0.6887
Macro F1: 0.5236
Weighted F1: 0.6386
Precision: 0.7228
Recall: 0.9060
F1: 0.8041


In [6]:
display(stage1_nb_report.head(10))


,class,precision,recall,f1_score,support
0,manipulative,0.722772,0.906028,0.804091,564
1,non_manipulative,0.430108,0.169492,0.243161,236


## Stage 2: Exact Technique Labels


In [7]:
try:
    stage2_majority_metrics, stage2_majority_predictions, stage2_majority_report = run_majority_baseline(stage2_df, stage_name='stage2_technique')
    stage2_nb_metrics, stage2_nb_predictions, stage2_nb_report = run_multinomial_nb(stage2_df, stage_name='stage2_technique', debug=False)
except ValueError as exc:
    if 'Stratified split requires at least 2 examples per class' not in str(exc):
        raise
    print('Rare exact technique labels caused the stratified split to fail. Re-running stage 2 with labels having <2 examples collapsed into OTHER.')
    stage2_majority_metrics, stage2_majority_predictions, stage2_majority_report = run_majority_baseline(stage2_df, stage_name='stage2_technique', rare_label_min_count=2)
    stage2_nb_metrics, stage2_nb_predictions, stage2_nb_report = run_multinomial_nb(stage2_df, stage_name='stage2_technique', debug=False, rare_label_min_count=2)

save_majority_metrics(stage2_majority_metrics, output_path=METRICS_DIR / 'stage2_technique_majority_baseline.json')
save_majority_predictions(stage2_majority_predictions, output_path=PREDICTIONS_DIR / 'stage2_technique_majority_baseline_predictions.csv')
stage2_majority_report.to_csv(METRICS_DIR / 'stage2_technique_majority_baseline_report.csv', index=False)

save_nb_metrics(stage2_nb_metrics, output_path=METRICS_DIR / 'stage2_technique_text_only_multinomial_nb.json')
save_nb_predictions(stage2_nb_predictions, output_path=PREDICTIONS_DIR / 'stage2_technique_text_only_multinomial_nb_predictions.csv')
save_classification_report(stage2_nb_report, output_path=METRICS_DIR / 'stage2_technique_text_only_multinomial_nb_report.csv')
save_baseline_comparison_chart(stage2_nb_metrics, majority_metrics_path=METRICS_DIR / 'stage2_technique_majority_baseline.json', output_path=FIGURES_DIR / 'stage2_technique_model_comparison.png')

print_majority_summary(stage2_majority_metrics)
print_nb_summary(stage2_nb_metrics)
print(f"Rare-label collapse threshold used: {stage2_nb_metrics['rare_label_min_count']}")
print(f"Collapsed labels: {stage2_nb_metrics['number_of_collapsed_labels']}")


Rare exact technique labels caused the stratified split to fail. Re-running stage 2 with labels having <2 examples collapsed into OTHER.
Majority Baseline Summary
Stage: stage2_technique
Total examples: 2154
Train size / test size: 1723 / 431
Number of classes: 78
Majority class: Persuasion or Seduction
Accuracy: 0.2668
Macro F1: 0.0074
Weighted F1: 0.1124
TF-IDF + Multinomial Naive Bayes Summary
Stage: stage2_technique
Total examples: 2154
Train size / test size: 1723 / 431
Accuracy: 0.2668
Macro F1: 0.0102
Weighted F1: 0.1299
Rare-label collapse threshold used: 2
Collapsed labels: 59


In [8]:
display(stage2_nb_report.head(20))


,class,precision,recall,f1_score,support
0,Persuasion or Seduction,0.276808,0.965217,0.430233,115
1,Shaming or Belittlement,0.111111,0.021277,0.035714,47
2,Intimidation,0.166667,0.043478,0.068966,46
3,Accusation,0.142857,0.027778,0.046512,36
4,Rationalization,0.000000,0.000000,0.000000,22
5,"Shaming or Belittlement,Accusation",0.000000,0.000000,0.000000,14
6,Evasion,0.000000,0.000000,0.000000,12
7,OTHER,0.000000,0.000000,0.000000,12
8,Brandishing Anger,0.000000,0.000000,0.000000,11
9,Denial,0.000000,0.000000,0.000000,11


## Interpretation Notes


In [9]:
print('Stage 1 is a straightforward binary benchmark.')
print('Stage 2 is much harder because the full exact Technique string creates many sparse classes.')
print('If stage 2 uses OTHER, that is only to make the stratified split possible; it is not the default target design.')


Stage 1 is a straightforward binary benchmark.
Stage 2 is much harder because the full exact Technique string creates many sparse classes.
If stage 2 uses OTHER, that is only to make the stratified split possible; it is not the default target design.
